# Workshop: late deliveries, from data to an honest number

Two hours, in class, together. You will take one small dataset through the whole
workflow of lessons 1 to 5: look at it, split it, build a pipeline, validate it,
compare two models, and test the winner once.

**How it works.** Each step has a short instruction, a code cell with blanks
written as `___`, and a check cell. Fill the blanks, run the code cell, then run
the check: it prints ✓ when the step is right, or ✗ with a hint. If you see
`NameError: name '___' is not defined`, a blank is still empty.

Each step ends with one question. Answer it in a sentence in the markdown cell
below it: the reasoning is the part the exam asks about.

**The data.** A courier company recorded 1,200 parcels: the route length, how
full the van was, the driver's experience, the departure time, the rain, the age
of the van, and whether the parcel arrived late. The rule that decides lateness
is known, and it is revealed at the end - so do not open `delivery_data.py` yet.

In [ ]:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import checks
from delivery_data import load_deliveries

X, y = load_deliveries()
X.head()

## Step 1. Look at the data, and find the number to beat

Print the size of the table, how many values are missing in each column, and the
share of late parcels. Then compute the **baseline**: the accuracy of a "model"
that always answers *on time*, ignoring every input.

In [ ]:
print(X.shape)
print(X.isna().sum())
print(y.value_counts(normalize=True))

baseline = ___   # accuracy of always answering "on time"
baseline

In [ ]:
checks.step1(baseline)

**Question.** Which column has gaps, and what share of it? Why is the baseline
the first number to know, before any model?

*Your answer:*

## Step 2. Split, before anything learns from the data

Keep a quarter of the parcels as a test set, and make sure both parts have the
same share of late parcels.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=___, stratify=___, random_state=0
)
print(len(X_train), len(X_test))

In [ ]:
checks.step2(X_train, X_test, y_train, y_test)

**Question.** What could go wrong if the test set had a different share of late
parcels from the training set?

*Your answer:*

## Step 3. One pipeline: fill the gaps, scale, classify

Build a pipeline that fills the missing rain readings with the median, puts every
column on the same scale, and fits a logistic regression. Do **not** fill or scale
`X_train` yourself: the pipeline does it, fold by fold.

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    ___,
    LogisticRegression(max_iter=1000),
)
model

In [ ]:
checks.step3(model, X_train)

**Question.** Why must the imputer and the scaler sit inside the pipeline, rather
than being applied to the whole table at the start?

*Your answer:*

## Step 4. Validate on the training set only

Estimate how well the model will do on new parcels with 5-fold cross-validation.
The test set stays untouched.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, ___, ___, cv=folds)
print(f"{scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
checks.step4(scores)

**Question.** Does the model beat the baseline of step 1, and by how much? What
does the ± tell you?

*Your answer:*

## Step 5. Could a curve do better?

First look at one column: the share of late parcels by hour of departure.

In [ ]:
late_by_hour = y_train.groupby(X_train["departure_hour"].round()).mean()
late_by_hour.plot(marker="o", ylabel="share late", xlabel="departure hour");

A straight line in `departure_hour` cannot follow that shape. Lesson 3's trick
applies: add squared terms as new columns and keep the same logistic regression.
Score the new model on **the same folds**.

In [ ]:
curved = make_pipeline(
    SimpleImputer(strategy="median"),
    PolynomialFeatures(degree=___, include_bias=False),
    StandardScaler(),
    LogisticRegression(max_iter=5000),
)
curved_scores = cross_val_score(curved, X_train, y_train, cv=folds)
print(f"straight {scores.mean():.3f} ± {scores.std():.3f}")
print(f"curved   {curved_scores.mean():.3f} ± {curved_scores.std():.3f}")

In [ ]:
checks.step5(curved_scores)

**Question.** Is the difference between the two models larger than the ± of
each? Which feature do you think the squared terms helped with, and why would
departure time behave like that?

*Your answer:*

## Step 6. One look at the test set

Choose the model that won the cross-validation, fit it on the whole training set,
and score it **once** on the test set.

In [ ]:
best = ___   # model or curved
best.fit(X_train, y_train)
test_accuracy = best.score(X_test, y_test)
print(f"test accuracy {test_accuracy:.3f}")

In [ ]:
checks.step6(test_accuracy)

**Question.** Is the test accuracy close to the cross-validated one? What would it
mean if it were much higher?

*Your answer:*

## How did it go?

In [ ]:
checks.summary()

## If you finish early

`vehicle_age_years` may or may not matter. Drop it, score the curved model again
on the same folds, and decide from the numbers whether it was helping. Write down
what you would need to see before calling a difference real.